In [ ]:
#game_1 : "find_this_mii"

import cv2
import numpy as np

# 讀入影片，並設定從第 4820 幀開始處理
cap = cv2.VideoCapture("wiiplay.mp4")
frame_seq = 4820
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_seq)
_, first_frame = cap.read()

# 設定模板的寬高，計算影像的中心點座標，並以第一幀影像中心偏下 35 像素作為模板(人物的臉部)
h, w = first_frame.shape[:2]
template_w, template_h = 50, 75
center_x, center_y = w // 2, h // 2
template = first_frame[center_y + 35 - template_h // 2 : center_y + 35 + template_h // 2,
                       center_x - template_w // 2 : center_x + template_w // 2]

# 將模板縮小到 80%，之後比較好偵測
template = cv2.resize(template, (0, 0), fx=0.8, fy=0.8)

# 更新模板尺寸（for 畫框用）
th, tw = template.shape[:2]

#匹配值大於 0.6 才會畫框
threshold = 0.6
while True:

    #每次迴圈從下一幀開始處裡
    frame_seq += 1
    if frame_seq > 5000:
        frame_seq = 4820 + 1  
    status_cap , frame = cap.set(cv2.CAP_PROP_POS_FRAMES , frame_seq)
    if not status_cap:
        break

    # 使用TM_CCOEFF_NORMED進行模板匹配，並儲存 result 中的最大值和位置
    result = cv2.matchTemplate(frame, template, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, max_loc = cv2.minMaxLoc(result)

    # 如果最大值大於threshold，就在儲存的位置上畫紅框
    if max_val > threshold:
        top_left = max_loc
        bottom_right = (top_left[0] + tw, top_left[1] + th)
        cv2.rectangle(frame, top_left, bottom_right, (0, 0, 255), 2)

    # 顯示影像，按esc跳出
    cv2.imshow("find_this_mii", frame)
    k = cv2.waitKey(1)
    if k == 27: 
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
#game_2 : "find_two_look_alike"

import cv2
import numpy as np
from ultralytics import YOLO

# 載入兩個 YOLO 模型，yolov8n.pt用於檢測行人，yolov8s-face-lindevs.pt用於檢測人臉
model_person = YOLO('yolov8n.pt')
model_face = YOLO('yolov8s-face-lindevs.pt')  

# 讀入影片並設定從第 2180 幀開始處理
cap = cv2.VideoCapture("wiiplay.mp4")
frame_seq = 2180

while True:
    
    #每次迴圈從下一幀開始處裡
    frame_seq += 1
    if frame_seq > 2380:
        frame_seq = 2180
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_seq)
    status_cap, frame = cap.read()
    if not status_cap:
        break

    # 使用 yolov8n 模型檢測行人，遍歷所有檢測到的框，如果是行人就畫綠色框
    results_person = model_person(frame)[0]
    for r in results_person.boxes:
        cls_id = int(r.cls[0])
        conf = float(r.conf[0])
        x1, y1, x2, y2 = map(int, r.xyxy[0])
        if cls_id == 0:  # class 0 表示行人
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)  

    # 使用 yolov8s-face 模型檢測人臉，並儲存所有人臉框座標和人臉影像，並在每個人臉上面畫藍色框
    results_face = model_face(frame)[0]
    face_boxes = []
    face_images = []
    for r in results_face.boxes:
        conf = float(r.conf[0])
        x1, y1, x2, y2 = map(int, r.xyxy[0])
        face_img = frame[y1:y2, x1:x2]
        face_boxes.append((x1, y1, x2, y2))
        face_images.append(face_img)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                

    # 使用 cv2.TM_CCOEFF_NORMED 找出 face_images 中最相似的兩張人臉
    if len(face_images) >= 2:
        max_similarity = -float('inf')
        idx1, idx2 = -1, -1
        for i in range(len(face_images)):
            for j in range(i + 1, len(face_images)):
                img1 = face_images[i]
                img2 = face_images[j]
                h1, w1 = img1.shape[:2]
                h2, w2 = img2.shape[:2]
                
                # 縮放較大的圖像以匹配較小圖像
                if h1 * w1 > h2 * w2:
                    img1 = cv2.resize(img1, (w2, h2), interpolation=cv2.INTER_AREA)
                else:
                    img2 = cv2.resize(img2, (w1, h1), interpolation=cv2.INTER_AREA)

                # 使用 TM_CCOEFF_NORMED 匹配人臉，並更新最大的相似度分數
                result = cv2.matchTemplate(img1, img2, cv2.TM_CCOEFF_NORMED)
                similarity = result[0][0]
                if similarity > max_similarity:
                    max_similarity = similarity
                    idx1, idx2 = i, j

        # 如果有找到，在最相似的兩張人臉上面畫紅框
        if idx1 != -1 and idx2 != -1:
            for idx in [idx1, idx2]:
                x1, y1, x2, y2 = face_boxes[idx]
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  # 紅框

    # 顯示影像，按esc跳出
    cv2.imshow("find_two_look_alike", frame)
    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [38]:
#game_3 : "find_the_fastest_character"
import cv2
import numpy as np
from ultralytics import YOLO

#載入 yolov8n 模型用於行人檢測
model = YOLO('yolov8n.pt')

#讀入影片並設定從第 2480 幀開始處理
video_path = "wiiplay.mp4"
start_frame = 2480

cap = cv2.VideoCapture(video_path)
frame_seq = start_frame-1

# trackers 儲存每個行人的追蹤器，prev_centers 儲存每個行人上一幀的中心點座標，total_movements 儲存每個行人的累計移動距離。
trackers = []
prev_centers = []
total_movements = []

while True:
    
    #每次迴圈從下一幀開始處理
    frame_seq += 1
    if frame_seq > 2600:
        frame_seq = start_frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_seq)
    status_cap, frame = cap.read()
    if not status_cap:
        break

    #在 display_frame 上面畫框
    display_frame = frame.copy()

     #檢查是否為起始幀，如果是的話，進行第一次偵測
    if frame_seq == start_frame:
        
        #使用 YOLOv8 檢測行人，並清空追蹤器、中心點和移動距離列表。
        results = model(frame)[0]
        trackers = []
        prev_centers = []
        total_movements = []

        #遍歷 YOLOv8 檢測到的框，如果是行人的話，畫籃框設定追蹤器並儲存中心點。
        for det in results.boxes.data:
            cls = int(det[5])
            if cls == 0:  # class 0 是 person
                
                #在行人上畫籃框
                cv2.rectangle(display_frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                
                #儲存框座標，並設定 Kernelized Correlation Filter 追蹤器
                x1, y1, x2, y2 = map(int, det[:4])
                tracker = cv2.TrackerKCF_create()
                tracker.init(frame, (x1, y1, x2 - x1, y2 - y1))
                trackers.append(tracker)
                
                #儲存中心點，用於後續距離計算
                center = ((x1 + x2) // 2, (y1 + y2) // 2)
                prev_centers.append(center)
                total_movements.append(0.0)

    #對於非起始幀，使用追蹤器更新位置。
    else:
        #更新每個追蹤器
        for i, tracker in enumerate(trackers):
            success, bbox = tracker.update(frame)
            
            #如果追蹤成功的話，儲存框座標並計算中心點
            if success:
                x, y, w, h = map(int, bbox)
                center = (x + w // 2, y + h // 2)

                # 畫綠框表示追蹤中
                cv2.rectangle(display_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

                # 使用歐氏距離公式計算兩幀間的移動距離，並累加移動距離然後更新中心點
                dx = center[0] - prev_centers[i][0]
                dy = center[1] - prev_centers[i][1]
                total_movements[i] += np.sqrt(dx ** 2 + dy ** 2)
                prev_centers[i] = center

    # 確保在起始幀後才標記最快角色
    if frame_seq > start_frame:
        if total_movements:
            
            #找出累計移動距離最大的行人
            fastest_idx = np.argmax(total_movements)
            tracker = trackers[fastest_idx]
            success, bbox = tracker.update(frame)
            
            #如果成功的話，在最快行人上畫紅色框
            if success:
                x, y, w, h = map(int, bbox)
                cv2.rectangle(display_frame, (x, y), (x + w, y + h), (0, 0, 255), 2)

    # 顯示影像，按esc跳出
    cv2.imshow("find_the_fastest_character", display_frame)
    key = cv2.waitKey(30)
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()



0: 384x640 13 persons, 2 fire hydrants, 1 skateboard, 54.1ms
Speed: 3.1ms preprocess, 54.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)


In [1]:
#game_4 : "find_two_odds"

import cv2
import numpy as np
from ultralytics import YOLO
import torch

# 載入 yolov8n 模型用於行人檢測
model = YOLO('yolov8n.pt')

# 讀入影片並設定從第 1650 幀開始處理
cap = cv2.VideoCapture("wiiplay.mp4")
frame_seq = 1650

# 用於光流計算的上一幀灰階圖
prev_gray = None

while True:

    #每次迴圈從下一幀開始處裡
    frame_seq += 1
    if frame_seq > 1800:
        frame_seq = 1650
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_seq)
    status_cap, frame = cap.read()
    if not status_cap:
        break
    
    # 用 YOLOv8 檢測行人，並儲存檢測框座標
    results = model(frame, classes=[0]) 
    bboxes = results[0].boxes.xyxy.cpu().numpy() 
    
    # 用目前幀和上一幀的灰階圖計算光流
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if prev_gray is not None:
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        # 每 20 像素，畫上一個藍色光流箭頭
        h, w = gray.shape
        step = 20 
        y, x = np.mgrid[0:h:step, 0:w:step].reshape(2, -1).astype(int)
        fx, fy = flow[y, x].T
        lines = np.vstack([x, y, x + fx, y + fy]).T.reshape(-1, 2, 2)
        lines = np.int32(lines + 0.5)
        
        for (x1, y1), (x2, y2) in lines:
            cv2.arrowedLine(frame, (x1, y1), (x2, y2), (255, 0, 0), 1, tipLength=0.3)
    
    #更新上一幀的灰階圖
    prev_gray = gray.copy()
    
    # directions 儲存每個行人的平均光流方向和框座標
    directions = []

    # 計算每個行人框內的光流平均方向
    for bbox in bboxes:
        x1, y1, x2, y2 = map(int, bbox[:4])
        bbox_flow = flow[y1:y2, x1:x2]
        mean_flow = np.mean(bbox_flow, axis=(0, 1))
        directions.append((mean_flow, (x1, y1, x2, y2)))
            
    
    # 找出兩個異常角色（朝向與大多數人相反）
    if len(directions) > 0:
        
        # 計算主要方向（取平均）
        mean_direction = np.mean([d[0] for d in directions], axis=0)
        
        # 計算每個框的流向與主要方向的差異
        diff_scores = [np.dot(d[0], mean_direction) / (np.linalg.norm(d[0]) * np.linalg.norm(mean_direction) + 1e-6) 
                      for d in directions]
        
        # 找到兩個餘弦相似度最低的
        odd_indices = np.argsort(diff_scores)[:2]
        
        # 繪製紅色矩形
        for idx in odd_indices:
            x1, y1, x2, y2 = directions[idx][1]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
    
    # 顯示影像，按esc跳出
    cv2.imshow("find_two_odds", frame)
    k = cv2.waitKey(1)
    if k == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 (no detections), 162.9ms
Speed: 18.1ms preprocess, 162.9ms inference, 11.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 70.9ms
Speed: 2.4ms preprocess, 70.9ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 59.3ms
Speed: 1.7ms preprocess, 59.3ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 83.6ms
Speed: 1.3ms preprocess, 83.6ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 75.1ms
Speed: 3.3ms preprocess, 75.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 71.9ms
Speed: 1.5ms preprocess, 71.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 48.5ms
Speed: 1.7ms preprocess, 48.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 59.4ms
Speed: 1.3ms preprocess, 59.4

In [1]:
#Game 5
#Use whatever you learned this semester to improve the result. Write a simple report on your method and observations.
#我先預處理了車牌影像：首先先放大影像 2 倍，並轉影像為黑白的，然後提升對比、去除噪點、最後裁減中間90%的區域。
#我將 OCR 每個位置的候選字元與信心值記錄起來，對每個位置選擇信心度最高且符合預期類型的字。
#使用 OCR 誤差對應字典，將常見偵測的錯字進行修正
#定義車牌格式為 2英文字 + 2數字 + 3英文字，如果符合格式就直接把車牌儲存起來，不再做偵測

from ultralytics import YOLO
import cv2
import pytesseract
import numpy as np
import re
from string import ascii_uppercase, digits

# 設定執行檔路徑
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# 讀入模型
car_detector = YOLO('yolov8n.pt')  # 用於檢測車輛
license_plate_detector = YOLO('license_plate_detector.pt')  # 用於檢測車牌

# 定義車牌格式：2 個英文 + 2 個數字 + 3 個英文
LICENSE_PLATE_PATTERN = r'^[A-Z]{2}\d{2}[A-Z]{3}$'
LETTERS = list(ascii_uppercase)
DIGITS = list(digits) 

# 近似有些英文和數字很像，用於處理常見 OCR 錯誤
CHAR_MAP = {'0': 'O', 'O': '0', '1': 'I', 'I': '1', '5': 'S', 'S': '5', '8': 'B', 'B': '8'}

# 儲存車牌的字典
license_plate_dict = {}

#預處理車牌圖像以提高 OCR 準確性
def preprocess_license_plate(image):
    
    # 放大圖像
    image = cv2.resize(image, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    # 轉為灰度
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # 應用 OTSU 閾值處理
    _, gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # 對比度增強
    gray = cv2.equalizeHist(gray)
    # 去噪
    processed_image = cv2.bilateralFilter(gray, 11, 17, 17)
    # 裁剪中間 90% 區域
    height, width = processed_image.shape[:2]
    crop_percentage = 0.9
    new_width = int(width * crop_percentage)
    new_height = int(height * crop_percentage)
    start_x = (width - new_width) // 2
    start_y = (height - new_height) // 2
    end_x = start_x + new_width
    end_y = start_y + new_height
    cropped_image = processed_image[start_y:end_y, start_x:end_x]

    return cropped_image

#校正車牌文字，選擇機率最高的英文和數字，並處理 8 個字以上的情況
def correct_license_plate(text, confidence_data):
    
    text = text.upper().replace(' ', '').strip()

    # 如果文字長度 >= 8，取後 7 個字
    if len(text) >= 8:
        text = text[-7:]
        print(f"Text trimmed to last 7 characters: {text}")  # 除錯用

    # 如果文字已符合格式，直接返回
    if len(text) == 7 and re.match(LICENSE_PLATE_PATTERN, text):
        return text
    
    # 校正 OCR 識別的車牌文字，確保符合 2 個英文 + 2 個數字 + 3 個英文
    corrected_text = [''] * 7
    for i in range(7):
        expected_type = 'letter' if i in [0, 1, 4, 5, 6] else 'digit'
        candidates = confidence_data.get(i, [])
        
        if candidates:
            # 過濾符合預期類型的字符
            valid_candidates = [(char, conf) for char, conf in candidates 
                              if (char in LETTERS if expected_type == 'letter' else char in DIGITS)]
            if not valid_candidates:
                # 嘗試使用相近的文字
                valid_candidates = [(CHAR_MAP.get(char, char), conf) for char, conf in candidates 
                                  if CHAR_MAP.get(char, char) in (LETTERS if expected_type == 'letter' else DIGITS)]
            
            if valid_candidates:
                # 選擇置信度最高的字符
                best_char, best_conf = max(valid_candidates, key=lambda x: x[1])
                corrected_text[i] = best_char
            else:
                # 從原始文字選擇（如果有效）
                if i < len(text):
                    char = CHAR_MAP.get(text[i], text[i])
                    corrected_text[i] = char if char in (LETTERS if expected_type == 'letter' else DIGITS) else ''
        else:
            # 無候選時，使用原始文字（如果有效）
            if i < len(text):
                char = CHAR_MAP.get(text[i], text[i])
                corrected_text[i] = char if char in (LETTERS if expected_type == 'letter' else DIGITS) else ''
    
    corrected_text = ''.join(corrected_text)
    
    # 驗證校正後的文字是否符合格式，符合的話回傳，不符合的話回傳原本的text
    if len(corrected_text) == 7 and re.match(LICENSE_PLATE_PATTERN, corrected_text):
        return corrected_text    
    return text

#車牌框座標生成唯一鍵，用於儲存和識別車牌
def get_plate_key(x1, y1, x2, y2):
    return f"{int(x1)}_{int(y1)}_{int(x2)}_{int(y2)}"

# 讀入car.mp4影片
cap = cv2.VideoCapture('car.mp4')

while True:
    ret, frame = cap.read()
    if not ret:
        break  # 影片結束

    # 5B: 使用 YOLOv8 檢測車輛（紅色矩形）
    car_results = car_detector(frame, classes=[2])  # class 2 是車輛
    car_bboxes = car_results[0].boxes.xyxy.cpu().numpy()

    # 繪製車輛的紅色矩形
    for car_bbox in car_bboxes:
        x1, y1, x2, y2 = map(int, car_bbox[:4])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  # 紅色矩形

        # 在汽車範圍內檢測車牌
        car_crop = frame[y1:y2, x1:x2, :]  # 裁剪汽車區域
        if car_crop.size == 0:
            continue
        license_plates = license_plate_detector(car_crop)[0]
        license_plate_bboxes = license_plates.boxes.xyxy.cpu().numpy()

        # 5C: 檢測車牌（藍色矩形）並進行 OCR
        for license_plate in license_plate_bboxes:
            lp_x1, lp_y1, lp_x2, lp_y2 = map(int, license_plate[:4])
            # 將車牌座標轉換回原始圖像座標
            global_lp_x1, global_lp_y1 = x1 + lp_x1, y1 + lp_y1
            global_lp_x2, global_lp_y2 = x1 + lp_x2, y1 + lp_y2

            # 生成車牌的唯一 key
            plate_key = get_plate_key(global_lp_x1, global_lp_y1, global_lp_x2, global_lp_y2)

            # 5D: 檢查車牌是否已儲存
            if plate_key in license_plate_dict and re.match(LICENSE_PLATE_PATTERN, license_plate_dict[plate_key]):
                # 若已儲存且格式正確，則使用儲存的車牌號碼
                corrected_text = license_plate_dict[plate_key]
                print(f"Using stored license plate: {corrected_text} for key {plate_key}")
            else:
                # 裁剪車牌區域
                license_plate_crop = frame[global_lp_y1:global_lp_y2, global_lp_x1:global_lp_x2, :]
                if license_plate_crop.size == 0:
                    continue

                # 預處理車牌圖像
                license_plate_crop_thresh = preprocess_license_plate(license_plate_crop)

                # 使用 Tesseract 進行文字識別，獲取置信度
                ocr_result = pytesseract.image_to_data(license_plate_crop_thresh, 
                                                     config='--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789', 
                                                     output_type=pytesseract.Output.DICT)
                text_list = ocr_result['text']
                confidences = ocr_result['conf']
                
                # 收集每個字符的置信度
                confidence_data = {}
                char_index = 0
                combined_text = ''
                for i, char in enumerate(text_list):
                    if char.strip() and confidences[i] != -1:
                        confidence_data[char_index] = confidence_data.get(char_index, []) + [(char.upper(), confidences[i])]
                        combined_text += char.upper()
                        char_index += 1

                # 校正車牌文字
                corrected_text = correct_license_plate(combined_text, confidence_data)

                # 儲存車牌
                if corrected_text:
                    license_plate_dict[plate_key] = corrected_text
                    print(f"Stored license plate: {corrected_text} for key {plate_key}")
                else:
                    print(f"Failed to correct license plate: {combined_text} (raw OCR output)")

            # 繪製車牌的藍色矩形
            cv2.rectangle(frame, (global_lp_x1, global_lp_y1), (global_lp_x2, global_lp_y2), (255, 0, 0), 2)  # 藍色矩形

            # 在車牌上方顯示識別的綠色文字
            if corrected_text:
                cv2.putText(frame, corrected_text, (global_lp_x1, global_lp_y1 - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
                print(f"Detected license plate number: {corrected_text}")

    # 顯示縮小後的影像，按esc退出
    frame = cv2.resize(frame, None, fx=0.3, fy=0.3)
    cv2.imshow('License Plate Detection', frame)
    k = cv2.waitKey(1)
    if k == 27: 
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 21 cars, 165.7ms
Speed: 11.8ms preprocess, 165.7ms inference, 21.7ms postprocess per image at shape (1, 3, 384, 640)

0: 608x640 1 license_plate, 84.4ms
Speed: 2.5ms preprocess, 84.4ms inference, 0.8ms postprocess per image at shape (1, 3, 608, 640)
Failed to correct license plate:  (raw OCR output)

0: 544x640 (no detections), 86.9ms
Speed: 2.5ms preprocess, 86.9ms inference, 0.9ms postprocess per image at shape (1, 3, 544, 640)

0: 608x640 1 license_plate, 74.5ms
Speed: 2.8ms preprocess, 74.5ms inference, 0.8ms postprocess per image at shape (1, 3, 608, 640)
Text trimmed to last 7 characters: NAI3NRU
Stored license plate: NA13NRU for key 988_1783_1185_1845
Detected license plate number: NA13NRU

0: 640x640 (no detections), 81.8ms
Speed: 2.4ms preprocess, 81.8ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 license_plate, 72.7ms
Speed: 3.1ms preprocess, 72.7ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)
Stored license p

6. (5pts) Any comments regarding the final exam? Which steps you believe you have completed? Which steps bother you?<br>

我覺得我完成了每一個作業的要求。最後要辨識車牌的時候我覺得最難，因為ocr常常出錯，要處理車牌影像、符合格式，如果偵測對的話要儲存起來不讓他再偵測錯誤的車牌，蠻難的。<br>

7. (5pts) Any suggestion to teaching assistants to improve this class? Any suggestion to teacher to improve this class?<br>

沒有其他建議，我在這門課學到很多影像視覺的東西，尤其第三節課通常會用作業自己練習，讓我可以更容易了解跟學習使用這些影像視覺的技術。<br>